This is a tutorial notebook for the signed network inference. The custom functions, mcmc sampler, can be found in the signed_inference.py modul

In [ ]:
#libraries needed
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
sns.set_style('ticks')
import sklearn
from sklearn.metrics import roc_curve
import scipy
import os
import multiprocessing
from multiprocessing import Pool
import random
from signed_inference import *
import datetime


The cell below runs the generative model to create synthetic data. $A$ is the signed network, P,N are the positive, and negative parts of it correspondingly, $X_s$ is the set of observation matrices, the variable num_of_obs controls the number of observations we have. The variable partitions consists of num_of_obs vectors to specify the partitions. The variable reals store the interaction probabilities $p_-,p_0,p_+$. The parameter $q$ for cross-group interactions is not specified, the default value is $0.05$. Observation period lengths are 250. The parameters $p_-,p_0,p_+$ are sampled from two possible distributions.

In [ ]:
num_of_obs=4

In [ ]:

g = nx.generators.random_graphs.erdos_renyi_graph(32, 0.5)

for e in g.edges():
    np.random.seed(int.from_bytes(os.urandom(4), byteorder='little'))
    g.edges[e]['weight'] = np.random.choice([-1, 1])

A = nx.adjacency_matrix(g).todense()
P = (A > 0).astype(int)
N = (A < 0).astype(int)
plt.subplot(1, 3, 1)
plt.title('A')
plt.imshow(A, vmin=-1, vmax=1, cmap=plt.cm.RdBu_r)
plt.subplot(1, 3, 2)
plt.title('P')
plt.imshow(P, vmin=-1, vmax=1, cmap=plt.cm.RdBu)
plt.subplot(1, 3, 3)
plt.title('N')
plt.imshow(N, vmin=-1, vmax=1, cmap=plt.cm.RdBu_r)

X_s=[]
#print(A)
partitions=[]
#rr={-1:0.25,0:0.655,1:0.9751}
reals=[]
lens=[]
for i in range(num_of_obs):
    leng=len(A)
    probs=np.ones(leng)
    probs=prob_list(leng)
    probs/=sum(probs)   
    partition=np.random.choice(range(leng),64,p=probs)
    #partition=np.zeros((len(A),1))
    np.random.seed(int.from_bytes(os.urandom(4), byteorder='little'))
    curr_len=np.random.randint(20,110)
    curr_len=250
    #print(partition)
    if np.random.rand()>0.5:
        rr1=np.random.uniform(0,0.2)
        rr2=np.random.uniform(0.4,0.5)
        rr3=np.random.uniform(0.7,0.75)
        print("+++")
    else:
        rr1=np.random.uniform(0.2,0.25)
        rr2=np.random.uniform(0.4,0.44)
        rr3=np.random.uniform(0.8,0.9)
    rr={-1:rr1,0:rr2,1:rr3}
    reals.append(rr)
    X_s.append(genrete_binomial_measurements(partition, A, curr_len, rates=rr,q_rate=0.1))
    partitions.append(partition)
    lens.append(curr_len)

In [ ]:
parta=np.zeros((len(X_s),len(X_s[0])))


The variable parta is the initialization for the partition variable for the MCMC sampler. It consists of num_of_obs partitions, where everyone is within the same group. num_of_steps, and num_of_chains specifies how many steps the mcmc shold make, and how many chains should be run.

In [ ]:
parta=np.zeros((len(X_s),len(X_s[0])))
num_of_steps=24000
num_of_chains=8

In [ ]:
X_s

In [ ]:
#Parallelized inference: this function helps paralizing the algorithm
def tryer_mcmc2(l):
    a,b,c,d=Joint_MCMC(X_s,lens,parta,l,1)
    print("rates:",b)
    print("partitions:",c)
    print('q_rates',d)
    return a,b

#we set the parameters for the mcmc samplers to run paralell
inputs=[]
for i in range(num_of_chains):
    inputs.append(num_of_steps)
#paralell run of inference algorithm
if __name__ == '__main__':
    with Pool(12) as p:
        #print(p.map(f, [1, 2, 3]))
        ll=p.map(tryer_mcmc2,inputs)


#visualization,etc...
inferences_mtx=[]
rates_inferred=[]
for i in ll:
    for j in i[0]:
        inferences_mtx.append(j)
    for k in i[1]:
        rates_inferred.append(k)
y=inferences_mtx
print(sum(y)/len(y))
y=sum(y)/len(y)
plt.imshow(A,cmap=plt.cm.bwr, vmin=-1, vmax=1)
plt.show()
plt.imshow(y,cmap=plt.cm.bwr, vmin=-1, vmax=1)
plt.colorbar()
plt.show()
positives=[]
negatives=[]
zeros=[]
#A=np.load('A.npy')
B=(A  <0).astype(int)
B=-B
#y=a
y=inferences_mtx
for i in y:
    negatives.append((i  <0).astype(int))
    positives.append((i  >0).astype(int))
    zeros.append((i==0).astype(int))
pos_mean=sum(positives)/len(positives)
zeros_mean=sum(zeros)/len(zeros)
plt.imshow(pos_mean, cmap=plt.cm.RdBu, vmin=-1, vmax=1)
plt.title("inferred positive edges")
plt.show()
plt.title("positive edges")
plt.imshow(A>0, cmap=plt.cm.RdBu, vmin=-1, vmax=1)
plt.show()
neg_mean=sum(negatives)/len(negatives)
plt.title("inferred negative edges")
plt.imshow(-neg_mean, cmap=plt.cm.RdBu, vmin=-1, vmax=1)

plt.show()

plt.title("real negative edges")
plt.imshow(B, cmap=plt.cm.RdBu, vmin=-1, vmax=1)
plt.show()
plt.title("inferred indifferent edges")
plt.imshow(zeros_mean,cmap=plt.cm.PRGn, vmin=-1, vmax=1)
plt.show()
plt.title("real indifferent edges")
plt.imshow(A==0, cmap=plt.cm.PRGn, vmin=-1, vmax=1)
plt.show()
pos_reals=[]
pos_inferred=[]
P=(A>0). astype(int)
N=(A<0). astype(int)
for j in range(len(P)):
    for i in range(len(P)): 
        pos_reals.append(P[i][j])


"""                   
for yy in pos_mean:
    for j in yy:
        pos_inferred.append(j)
"""
for j in range(len(pos_mean)):
    for i in range(len(pos_mean)):
       pos_inferred.append(pos_mean[i][j])
#print(len(pos_inferred),'pozitivak',pos_reals,pos_inferred)
fpr_pos,tpr_pos,thresholds=sklearn.metrics.roc_curve(pos_reals,pos_inferred)

#print(pos_inferred)
neg_reals=[]
neg_inferred=[]
for xx in N:
    for i in xx:
        neg_reals.append(i)
for yy in neg_mean:
    for j in yy:
        neg_inferred.append(j)
for j in range(len(N)):
    for i in range(len(N)):
        neg_reals.append(N[i][j])

for j in range(len(neg_mean)):
    for i in range(len(neg_mean)):
        neg_inferred.append(neg_mean[i][j])
    
fpr_neg,tpr_neg,thresholds=sklearn.metrics.roc_curve(neg_reals,neg_inferred)



print(len(pos_inferred),"alappontok száma")



print("AUC score for positive edges: ",sklearn.metrics.roc_auc_score(pos_reals, pos_inferred) )
print("AUC score for negative edges: ",sklearn.metrics.roc_auc_score(neg_reals, neg_inferred) )

plt.subplot(2,2,1)
plt.title("ROC curve positive edges")

plt.plot(fpr_pos,tpr_pos)
plt.plot([0,1],[0,1],"--r")
plt.text(0.5,0,"AUC: "+str(np.round(sklearn.metrics.roc_auc_score(pos_reals, pos_inferred),4)))
#plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

plt.subplot(2,2,2)
plt.title("ROC curve negative edges")
plt.plot(fpr_neg,tpr_neg)
plt.text(0.5,0,"AUC: "+str(np.round(sklearn.metrics.roc_auc_score(neg_reals, neg_inferred),4)))
plt.plot([0,1],[0,1],"--r")
#plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.savefig("combined.svg",bbox_inches='tight')
plt.show()


#print('negatív',neg_reals,neg_inferred)